# Torch vs ONNX performance

In [10]:
import torch
import onnx
import onnxruntime
import lenet
import numpy as np

torch.manual_seed(42)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Test dataset

In [11]:
X = lenet.load_tensor(lenet.sample_filename)
y = lenet.load_tensor(lenet.labels_filename, dtype="<i8")

X.shape, y.shape

((64, 1, 28, 28), (64,))

## PyTorch model

In [12]:
lenet_model = lenet.LeNetModel().to(device)
lenet_model.load_state_dict(torch.load(lenet.torch_filename, map_location=device))
lenet_model.eval()


LeNetModel(
  (lenet_stack): Sequential(
    (0): Conv2d(1, 12, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1152, out_features=120, bias=True)
    (8): ReLU()
    (9): Linear(in_features=120, out_features=84, bias=True)
    (10): Sigmoid()
    (11): Linear(in_features=84, out_features=10, bias=True)
  )
)

## ONNX

In [13]:
onnx_model = onnx.load(lenet.onnx_filename)
onnx.checker.check_model(onnx_model)
print(f"onnx op_set version={onnx.defs.onnx_opset_version()}")
print(f"onnx_model.op_set_version={onnx_model.opset_import[0].version}")
print(f"onnx_model.ir_version={onnx_model.ir_version}")

model_input = onnx_model.graph.input[0]
print(f"onnx_model.input: name={model_input.name}, type={model_input.type.tensor_type.elem_type} shape={model_input.type.tensor_type.shape.dim}")

model_output = onnx_model.graph.output[0]
print(f"onnx_model.output: name={model_output.name}, type={model_output.type.tensor_type.elem_type} shape={model_output.type.tensor_type.shape.dim}")


onnx op_set version=28
onnx_model.op_set_version=20
onnx_model.ir_version=10
onnx_model.input: name=images, type=1 shape=[dim_value: 1
, dim_value: 1
, dim_value: 28
, dim_value: 28
]
onnx_model.output: name=logits, type=1 shape=[dim_value: 1
, dim_value: 10
]


## ONNX C++ inference

In [14]:
import subprocess
from pathlib import Path

project_root = Path(lenet.__file__).resolve().parent.parent
subprocess.run(
    [str(project_root / "build/onnx_exporter"), "model/lenet.onnx"],
    cwd=project_root,
    check=True,
)


ONNX Runtime v1.30.1
Available providers:
  CPUExecutionProvider
Running inference
[-3.205126-3.205126,-1.3334465-1.3334465,-1.8047261-1.8047261,0.449992840.44999284,-6.5404344-6.5404344,1.47316271.4731627,-4.8560433-4.8560433,5.3006815.300681,-0.21304229-0.21304229,12.73088612.730886]
[3.13816123.1381612,-2.509291-2.509291,11.85114311.851143,-1.0311589-1.0311589,4.3142944.314294,-5.834595-5.834595,4.43119144.4311914,-9.748271-9.748271,0.89651510.8965151,-4.657622-4.657622]
[0.91011630.9101163,12.20722112.207221,-2.989013-2.989013,2.62502962.6250296,0.763601360.76360136,-4.4869723-4.4869723,1.75267311.7526731,-3.9990015-3.9990015,-1.9131021-1.9131021,-2.8173313-2.8173313]
[2.68416672.6841667,13.51095713.510957,-0.9756792-0.9756792,4.25860744.2586074,0.057496450.05749645,-3.3802953-3.3802953,-0.25939542-0.25939542,-5.493281-5.493281,-4.1746893-4.1746893,-4.9729633-4.9729633]
[9.5321459.532145,-3.592979-3.592979,3.5602833.560283,1.50392851.5039285,-0.05797252-0.05797252,-2.114326-2.11432

CompletedProcess(args=['/home/arykovanov/src/learning/arm_preparation/build/onnx_exporter', 'model/lenet.onnx'], returncode=0)

In [15]:
cpp_logits = lenet.load_tensor(lenet.cpp_logits_filename)
cpp_logits.shape

(64, 10)

## Logit values

Run inference with both PyTorch and ONNX Runtime for all the presaved image batch

In [16]:
onnxruntime_session = onnxruntime.InferenceSession(lenet.onnx_filename)

torch_logits = torch.zeros((len(X), 10))
onnx_logits = torch.zeros((len(X), 10))

with torch.no_grad():
    for idx in range(len(y)):
        img = X[idx]
        label = y[idx]
        pred_onnx = onnxruntime_session.run(None, {"images": img.reshape(1, *img.shape)})[0].reshape(-1)
        pred_torch = lenet_model(torch.from_numpy(img.reshape(1, 1, 28, 28)))
        torch_logits[idx] = pred_torch
        onnx_logits[idx] = torch.from_numpy(pred_onnx)


Check if ONNX and Pytorch prediction are the same.

In [17]:
print(f"label: {y[0]}")
print(f"torch: {torch_logits[0]}")
print(f"onnx: {onnx_logits[0]}")
print(f"cpp: {cpp_logits[0]}")

for idx in range(len(y)):
    label = y[idx]
    # print(np.abs(onnx_logits[idx] - torch_logits[idx]).max())
    if torch_logits[idx].argmax() != onnx_logits[idx].argmax():
        print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={torch_logits[idx].argmax()}, pred_onnx={onnx_logits[idx].argmax()}")
    if torch_logits[idx].argmax() != cpp_logits[idx].argmax():
        print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={torch_logits[idx].argmax()}, pred_cpp={cpp_logits[idx].argmax()}")
    if abs(onnx_logits[idx] - torch_logits[idx]).max() > 1e-5:
        raise ValueError(f"Logits at sample {idx}: label={label}, pred_torch={torch_logits[idx]}, pred_onnx={onnx_logits[idx]}. The difference is {np.abs(onnx_logits[idx] - torch_logits[idx]).max()} and it is TOO BIG! The difference should be less than 1e-5.")


label: 9
torch: tensor([-3.2051, -1.3334, -1.8047,  0.4500, -6.5404,  1.4732, -4.8560,  5.3007,
        -0.2130, 12.7309])
onnx: tensor([-3.2051, -1.3334, -1.8047,  0.4500, -6.5404,  1.4732, -4.8560,  5.3007,
        -0.2130, 12.7309])
cpp: [-3.205126   -1.3334465  -1.8047261   0.44999284 -6.5404344   1.4731627
 -4.8560433   5.300681   -0.21304229 12.730886  ]
